# 03 · Measured end-to-end latency: detection → SIEM → automated response

This notebook answers Reviewer 1 (comment 1, final part) and Reviewer 2 (comments 3 and 6). The CICIDS-2018 test stream, which has **genuine native timestamps**, is replayed event by event in timestamp order. Each pipeline stage is timed on the actual hardware:

`TAE encode → window assembly → HBLT classify → ECS JSON → SIEM ingest → automated response action`

* **SIEM ingest** has two modes. Direct Elasticsearch uses `refresh=wait_for`, so the call returns only once the alert is searchable and therefore visible in Kibana. Logstash HTTP input → Elasticsearch polls until the alert is searchable.
* **Automated response** has two modes: a blocklist file with fsync, or a POST to a SOAR webhook. This is a testbed **automated containment latency**. It is *not* a human incident-response time.
* The measured service times are combined with the real arrival times through a FIFO queue, so queueing under the observed traffic rate is included. Load tests replay the stream 10× and 100× faster.
* CICIDS-2018 timestamps have one-second resolution, so events that share a second are treated as simultaneous arrivals. This makes the queueing estimate conservative.
* End-to-end detection latency for an attack event = (time the alert becomes searchable) − (arrival of the attack event), under the **causal** matching rule. It contains the event-time delay, window accumulation, queueing, inference, JSON generation and indexing.

### Starting the SIEM stack
```bash
cd docker && docker compose up -d                     # Elasticsearch 8, Logstash (HTTP input :8080), Kibana :5601
export ES_URL=http://localhost:9200 LOGSTASH_URL=http://localhost:8080
# optional: export SOAR_WEBHOOK_URL=http://<your-soar>/webhook
```
Without these variables the notebook still measures model, JSON and file-sink latencies, and reports that SIEM ingestion was not measured.

In [1]:
import os, sys, json, warnings
from pathlib import Path
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
ROOT = Path.cwd() if (Path.cwd() / "talstmae").exists() else Path.cwd().parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import tensorflow as tf
from talstmae import Config, set_all_seeds
from talstmae import data as DT, models as M, metrics as MT, experiment as X, alerts as AL

SMOKE_TEST = False      # True = tiny synthetic data, ONLY to check that the code executes. Never report its numbers.
RESUME = True           # continue an interrupted run (finished models/seeds are skipped)
cfg = Config().smoke() if SMOKE_TEST else Config()
if os.environ.get("TALSTMAE_SETTINGS"):          # settings written by 00_RUN_ME.ipynb override the lines above
    _s = json.loads(Path(os.environ["TALSTMAE_SETTINGS"]).read_text())
    _base = Config(**{k: (tuple(v) if k in ("datasets", "seeds") else v) for k, v in _s.get("config", {}).items()})
    SMOKE_TEST, RESUME = bool(_s.get("smoke_test")), bool(_s.get("resume", True))
    cfg = _base.smoke() if SMOKE_TEST else _base
    print("settings from 00_RUN_ME:", _s)
if SMOKE_TEST:
    print("=" * 78 + "\nSMOKE TEST MODE: synthetic placeholder data. Results are INVALID and must not be reported.\n" + "=" * 78)
set_all_seeds(cfg.seed, cfg.deterministic_ops)
print("TensorFlow", tf.__version__, "| outputs ->", cfg.out.resolve())

import re
from talstmae import latency as LT, baselines as B
HW = LT.hardware_info(); HW

I0000 00:00:1794275467.477148     142 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


I0000 00:00:1794275468.379755     142 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


settings from 00_RUN_ME: {'smoke_test': False, 'resume': True, 'latency_events': 5000, 'config': {'datasets': ['cicids2018'], 'seeds': [42, 43, 44]}}
TensorFlow 2.21.0 | outputs -> /tmp/claude-0/-home-claude/5b43f487-a08c-5b08-be88-1e2ff98cfe79/scratchpad/srv/ta-lstmae/outputs


{'cpu': 'Intel(R) Xeon(R) Processor @ 2.10GHz',
 'logical_cores': 2,
 'platform': 'Linux-6.18.44-fc-v338-x86_64-with-glibc2.39',
 'gpus': [],
 'tensorflow': '2.21.0'}

## 1. Models (seed-42 checkpoints from notebook 01; trained here if missing)

In [2]:
DS = "cicids2018"                                   # genuine timestamps; NSL-KDD/UNSW-NB15 can be added as labelled synthetic cases
D = DT.prepare_dataset(DS, cfg)
d = cfg.out / DS
if (d / "tae_encoder.keras").exists() and (d / "hblt_full.keras").exists():
    enc = tf.keras.models.load_model(d / "tae_encoder.keras")
    clf = tf.keras.models.load_model(d / "hblt_full.keras")
    print("loaded seed-42 checkpoints from notebook 01")
else:
    _, enc, lat = X.train_tae(D, cfg, cfg.seed, save_dir=d)
    clf = X.train_window_classifier(D, lat, cfg, M.CONFIGS[M.FULL]["branches"], cfg.hblt_epochs[DS], cfg.seed)[0]
    clf.save(d / "hblt_full.keras")
lat = {k: enc.predict(D[f"X_{k}"], batch_size=4096, verbose=0) for k in ("tr", "val", "te")}
clf_causal = X.train_window_classifier(D, lat, cfg, M.CONFIGS[M.CAUSAL]["branches"], cfg.hblt_epochs[DS], cfg.seed)[0]
clf_cnn = X.train_window_classifier(D, lat, cfg, M.CONFIGS["TAE + CNN only"]["branches"], cfg.hblt_epochs[DS], cfg.seed)[0]
hae = X.train_hae_hrl(D, cfg, cfg.seed)
PIPELINES = {"TA-LSTMAE (full)": LT.WindowPipeline(enc, clf, cfg.seq_len, "TA-LSTMAE"),
             "TA-LSTMAE (TAE + CNN)": LT.WindowPipeline(enc, clf_cnn, cfg.seq_len, "TA-LSTMAE-CNN"),
             "TA-LSTMAE causal": LT.WindowPipeline(enc, clf_causal, cfg.seq_len, "TA-LSTMAE-causal"),
             "HAE-HRL (re-implemented)": LT.RecordPipeline(hae["models"][1], hae["models"][0], "HAE-HRL")}
MODELS_FOR_THROUGHPUT = {"TA-LSTMAE (full)": (enc, clf), "TA-LSTMAE (TAE + CNN)": (enc, clf_cnn),
                         "TA-LSTMAE causal": (enc, clf_causal), "HAE-HRL (re-implemented)": tuple(hae["models"])}

[cicids2018]


  CICIDS-2018 attack categories: ['Benign', 'FTP-BruteForce', 'SSH-Bruteforce']
  CICIDS-2018 subset (1/6)       174,763  (expected 174,763)  OK


  input_dim=78 OK | train=110,102 val=12,233 test=52,428 | class_weight={0: 0.7375, 1: 1.5526} | timestamps: native flow timestamps (chronological split), resolution=1.0s


loaded seed-42 checkpoints from notebook 01


## 2. Sinks and response actions

In [3]:
ES_URL, LS_URL, SOAR = os.environ.get("ES_URL"), os.environ.get("LOGSTASH_URL"), os.environ.get("SOAR_WEBHOOK_URL")
LDIR = cfg.out / "latency"; LDIR.mkdir(exist_ok=True)
SINKS = {"file (no SIEM)": lambda: LT.FileSink(LDIR / "alerts_replay.ndjson")}
if ES_URL: SINKS["Elasticsearch (refresh=wait_for)"] = lambda: LT.ElasticsearchSink(ES_URL)
if ES_URL and LS_URL: SINKS["Logstash -> Elasticsearch"] = lambda: LT.LogstashSink(LS_URL, ES_URL)
RESPONDER = (lambda: LT.WebhookResponder(SOAR)) if SOAR else (lambda: LT.FileBlocklistResponder(LDIR / "blocklist.jsonl"))
print("sinks:", list(SINKS), "| response action:", "SOAR webhook" if SOAR else "blocklist file (fsync)")
if not ES_URL:
    warnings.warn("ES_URL not set: SIEM ingestion latency is NOT measured in this run.")

sinks: ['file (no SIEM)'] | response action: blocklist file (fsync)


/tmp/ipykernel_142/664800497.py:9: UserWarning: ES_URL not set: SIEM ingestion latency is NOT measured in this run.
  warnings.warn("ES_URL not set: SIEM ingestion latency is NOT measured in this run.")


## 3. Replay the test stream and time every stage
Set `N_EVENTS = None` to replay the complete test stream (52,428 events for CICIDS-2018).

In [4]:
N_EVENTS = 300 if cfg.smoke_test else globals().get("_s", {}).get("latency_events", 5000)   # None = full test stream
START = cfg.seq_len - 1
REPS, SUMMARY = {}, []
for pname, pipe in PIPELINES.items():
    for sname, make_sink in SINKS.items():
        rep = LT.replay(pipe, D["X_te"], D["y_te"], D["ts_te"], DS, cfg.threshold, sink=make_sink(),
                        responder=RESPONDER(), start=START, n_events=N_EVENTS)
        e2e = LT.end_to_end_alerts(rep, cfg.match_tolerance)
        REPS[(pname, sname)] = (rep, e2e)
        rep.to_csv(LDIR / ("replay_" + re.sub(r"[^A-Za-z0-9]+", "_", f"{pname}_{sname}").strip("_") + ".csv"), index=False)
        s = LT.summarize_latency(rep, e2e); s.insert(0, "sink", sname); s.insert(0, "model", pname)
        SUMMARY.append(s)
        print(f"{pname:<26s} {sname:<34s} alerts={int(rep.y_pred.sum()):,}  detected attacks={len(e2e):,}")
SUMMARY = pd.concat(SUMMARY); SUMMARY.to_csv(LDIR / "latency_summary.csv", index=False)
SUMMARY.round(3)

TA-LSTMAE (full)           file (no SIEM)                     alerts=2,591  detected attacks=2,594


TA-LSTMAE (TAE + CNN)      file (no SIEM)                     alerts=2,596  detected attacks=2,594


TA-LSTMAE causal           file (no SIEM)                     alerts=2,594  detected attacks=2,594


HAE-HRL (re-implemented)   file (no SIEM)                     alerts=2,596  detected attacks=2,594


,model,sink,stage,n,mean_ms,median_ms,p95_ms,p99_ms,max_ms
0,TA-LSTMAE (full),file (no SIEM),TAE encode,5000,1.597,1.528,2.076,2.939,7.608
1,TA-LSTMAE (full),file (no SIEM),window assembly,5000,0.023,0.022,0.030,0.049,0.817
2,TA-LSTMAE (full),file (no SIEM),classify,5000,2.060,1.967,2.657,3.739,8.278
3,TA-LSTMAE (full),file (no SIEM),ECS JSON (alerts),2591,0.055,0.051,0.076,0.096,0.548
4,TA-LSTMAE (full),file (no SIEM),SIEM ingest (alerts),2591,0.017,0.015,0.026,0.047,0.622
5,TA-LSTMAE (full),file (no SIEM),response action (alerts),2591,0.322,0.295,0.466,0.673,21.021
6,TA-LSTMAE (full),file (no SIEM),per-event service,5000,3.885,3.760,4.976,6.733,25.062
7,TA-LSTMAE (full),file (no SIEM),end-to-end detection,2594,13.647,12.532,26.794,32.271,47.560
8,TA-LSTMAE (full),file (no SIEM),automated response,2594,13.969,12.853,27.178,32.663,47.901
9,TA-LSTMAE (full),file (no SIEM),event-time delay (causal),2594,0.000,0.000,0.000,0.000,0.000


## 4. Load test: the same measured service times under 1×, 10× and 100× the observed arrival rate

In [5]:
LOAD = []
for (pname, sname), (rep, _) in REPS.items():
    gaps = np.diff(rep.ts.values); rate = 1 / gaps[gaps > 0].mean() if np.any(gaps > 0) else np.nan
    for sp in (1, 10, 100):
        e2e = LT.end_to_end_alerts(rep, cfg.match_tolerance, speedup=sp)
        LOAD.append(dict(model=pname, sink=sname, speedup=sp, arrival_rate_eps=rate * sp,
                         utilisation=rep.t_service.mean() * rate * sp,
                         e2e_detection_median_ms=e2e.end_to_end_detection_s.median() * 1e3,
                         e2e_detection_p95_ms=e2e.end_to_end_detection_s.quantile(0.95) * 1e3,
                         automated_response_median_ms=e2e.automated_response_s.median() * 1e3))
LOAD = pd.DataFrame(LOAD); LOAD.to_csv(LDIR / "latency_load_test.csv", index=False); LOAD.round(3)

,model,sink,speedup,arrival_rate_eps,utilisation,e2e_detection_median_ms,e2e_detection_p95_ms,automated_response_median_ms
0,TA-LSTMAE (full),file (no SIEM),1,1.0,0.004,12.532,26.794,12.853
1,TA-LSTMAE (full),file (no SIEM),10,10.0,0.039,13.063,28.577,13.415
2,TA-LSTMAE (full),file (no SIEM),100,100.0,0.388,6438.451,13944.420,6438.768
3,TA-LSTMAE (TAE + CNN),file (no SIEM),1,1.0,0.003,8.960,19.066,9.269
4,TA-LSTMAE (TAE + CNN),file (no SIEM),10,10.0,0.027,8.960,19.066,9.269
5,TA-LSTMAE (TAE + CNN),file (no SIEM),100,100.0,0.268,3745.724,8176.344,3746.092
6,TA-LSTMAE causal,file (no SIEM),1,1.0,0.003,10.985,23.239,11.288
7,TA-LSTMAE causal,file (no SIEM),10,10.0,0.033,11.162,23.479,11.463
8,TA-LSTMAE causal,file (no SIEM),100,100.0,0.330,5372.345,11154.760,5372.662
9,HAE-HRL (re-implemented),file (no SIEM),1,1.0,0.004,12.346,26.174,12.743


## 5. Offline batched throughput (events/s)

In [6]:
TP = pd.concat([LT.throughput(mods, D["X_te"], cfg.seq_len, (1, 32, 512)).assign(model=name)
                for name, mods in MODELS_FOR_THROUGHPUT.items()], ignore_index=True)
TP.to_csv(LDIR / "throughput.csv", index=False)
(LDIR / "hardware.json").write_text(json.dumps(HW, indent=2)); TP

,batch_size,events,seconds,events_per_second,model
0,1,2000,5.273653,379.243777,TA-LSTMAE (full)
1,32,2000,0.670011,2985.023792,TA-LSTMAE (full)
2,512,10240,0.850659,12037.732060,TA-LSTMAE (full)
3,1,2000,2.201191,908.599042,TA-LSTMAE (TAE + CNN)
4,32,2000,0.401607,4979.991068,TA-LSTMAE (TAE + CNN)
5,512,10240,0.323960,31608.878552,TA-LSTMAE (TAE + CNN)
6,1,2000,3.731868,535.924673,TA-LSTMAE causal
7,32,2000,0.425582,4699.451425,TA-LSTMAE causal
8,512,10240,0.426359,24017.339018,TA-LSTMAE causal
9,1,2000,4.040438,494.995830,HAE-HRL (re-implemented)


## 6. Reporting guidance
* Report the **causal event-time MTTD** (notebook 01) and the **measured end-to-end detection latency** (median and p95) separately. Also report the per-stage breakdown and the hardware used.
* Replace "projected MTTR" by the **measured automated-response latency** only for the testbed action implemented here, and describe it as such. Human or analyst MTTR remains out of scope.
* Make claims of "SIEM integration" only for the sinks actually measured in this run, i.e. Elasticsearch and/or Logstash.